#Test DeepSeek-R1-Distill-Qwen-7B

In [ ]:
%%capture
!pip install opentelemetry-api==1.38.0 opentelemetry-sdk==1.38.0 opentelemetry-proto==1.38.0 opentelemetry-exporter-otlp-proto-common==1.38.0

In [ ]:
!pip install -q transformers accelerate bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.9 MB/s eta 0:00:00


In [13]:
# Cell 1: Xóa model
del model
del tokenizer

# Cell 2: Dọn sạch
import torch, gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

# Cell 3: Kiểm tra
used = torch.cuda.memory_allocated() / 1e9
free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
print(f"Used : {used:.2f} GB")
print(f"Free : {free:.2f} GB")

Used : 0.01 GB
Free : 15.63 GB


In [14]:
import torch, gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
print(f"Free: {free:.1f}GB")

Free: 15.6GB


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"  # chỉ ~1.5GB

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb,
    device_map="auto"
)
print(f"✅ Loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

✅ Loaded | VRAM: 5.6GB


In [ ]:
def chat(user_msg, system_msg=None, max_new_tokens=400):
    messages = []
    if system_msg:
        messages.append({"role": "system", "content": system_msg})
    messages.append({"role": "user", "content": user_msg})

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.6,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    # Tách phần <think> và phần trả lời
    import re
    think = re.search(r'<think>(.*?)</think>', response, re.DOTALL)
    answer = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL).strip()

    if think:
        print("🧠 <think>")
        print(think.group(1).strip()[:600])
        print("</think>\n")
    print("💬 Trả lời:")
    print(answer)
    return response

In [ ]:
chat("ừ thôi kệ đi",
     system_msg="""Bạn là chuyên gia phân tích cảm xúc trong hội thoại chăm sóc khách hàng (CSKH) bằng tiếng Việt. Bạn chỉ nói tiếng Việt không nói tiếng Trung.

Nhiệm vụ của bạn là phân tích câu nói của khách hàng và xác định cảm xúc thực sự của họ dựa trên ngữ nghĩa, ngữ cảnh giao tiếp và sắc thái lời nói.

Hãy thực hiện các bước sau:

1. Xác định cảm xúc chính của câu nói. Chọn một trong các nhãn sau:

* Rất tiêu cực
* Tiêu cực
* Trung tính
* Tích cực
* Rất tích cực

2. Giải thích lý do:

* Phân tích các từ khóa hoặc cách diễn đạt thể hiện cảm xúc.
* Xem xét sắc thái mỉa mai, chán nản, bực bội hoặc miễn cưỡng nếu có.
* Nếu câu nói có thể mang nhiều nghĩa, hãy giải thích nghĩa hợp lý nhất trong bối cảnh CSKH.

3. Đánh giá thái độ của khách hàng đối với dịch vụ:

* Hài lòng
* Không hài lòng
* Thờ ơ / miễn cưỡng
* Đang bực bội
* Đang thất vọng

4. Đưa ra mức độ cảm xúc theo thang điểm từ -2 đến +2:

* -2: rất tiêu cực
* -1: tiêu cực
* 0: trung tính
* +1: tích cực
* +2: rất tích cực

5. Trả lời theo định dạng JSON sau:

{
"emotion_label": "...",
"sentiment_score": ...,
"customer_attitude": "...",
"explanation": "..."
}

Chỉ phân tích dựa trên nội dung câu nói, không suy đoán quá mức nếu thiếu thông tin.
""")

💬 Trả lời:
Okay, I'm trying to figure out how to respond to this user's query. They provided a detailed problem where they want me to act as a Chinese expert analyzing feedback in a video call setting (CSKH) using Vietnamese. The user specified that I should only respond in Vietnamese and not use any Chinese characters. They also mentioned that my job is to analyze the feedback, determine the sentiment, and then provide a JSON response with emotion labels, sentiment scores, customer attitudes, and explanations.

First, I need to understand the problem fully. The user is asking me to act as an expert in analyzing feedback from a video call, but they want the response in Vietnamese. They also provided an example response, which is helpful. I should make sure to follow the example's structure and guidelines.

Next, I should consider the user's potential needs. They might be working on a project that involves analyzing customer feedback in a video call setting, possibly for a marketing, cu

'Okay, I\'m trying to figure out how to respond to this user\'s query. They provided a detailed problem where they want me to act as a Chinese expert analyzing feedback in a video call setting (CSKH) using Vietnamese. The user specified that I should only respond in Vietnamese and not use any Chinese characters. They also mentioned that my job is to analyze the feedback, determine the sentiment, and then provide a JSON response with emotion labels, sentiment scores, customer attitudes, and explanations.\n\nFirst, I need to understand the problem fully. The user is asking me to act as an expert in analyzing feedback from a video call, but they want the response in Vietnamese. They also provided an example response, which is helpful. I should make sure to follow the example\'s structure and guidelines.\n\nNext, I should consider the user\'s potential needs. They might be working on a project that involves analyzing customer feedback in a video call setting, possibly for a marketing, cust

# Test Qwen3-8B

In [ ]:
# Cell 2: Load Qwen3-8B với config tiết kiệm hơn
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch, os

# Giúp tránh fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,   # double quant tiết kiệm thêm ~0.5GB
)

model_name = "Qwen/Qwen3-8B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb,
    device_map="auto",
    low_cpu_mem_usage=True,   # ← thêm dòng này
)
print(f"✅ VRAM used: {torch.cuda.memory_allocated()/1e9:.1f}GB")

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

✅ VRAM used: 6.1GB


In [ ]:
def chat_qwen3(user_msg, system_msg=None, thinking=True, max_new_tokens=512):
    # /think = bật reasoning | /no_think = trả lời ngay
    prefix = "/think\n" if thinking else "/no_think\n"

    messages = []
    if system_msg:
        messages.append({"role": "system", "content": system_msg})
    messages.append({"role": "user", "content": prefix + user_msg})

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    import re
    think = re.search(r'<think>(.*?)</think>', response, re.DOTALL)
    answer = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL).strip()

    if think:
        print("🧠 <think>")
        print(think.group(1).strip()[:600])
        print("</think>\n")
    print("💬 Trả lời:")
    print(answer)
    return answer

In [ ]:
msg = "Khách nói: 'ừ thôi kệ đi' sau 5 lần hỏi giao hàng. Cảm xúc là gì?"
sys = """Bạn là chuyên gia phân tích cảm xúc trong hội thoại chăm sóc khách hàng (CSKH) bằng tiếng Việt. Bạn chỉ nói tiếng Việt không nói tiếng Trung.

Nhiệm vụ của bạn là phân tích câu nói của khách hàng và xác định cảm xúc thực sự của họ dựa trên ngữ nghĩa, ngữ cảnh giao tiếp và sắc thái lời nói.

Hãy thực hiện các bước sau:

1. Xác định cảm xúc chính của câu nói. Chọn một trong các nhãn sau:

* Rất tiêu cực
* Tiêu cực
* Trung tính
* Tích cực
* Rất tích cực

2. Giải thích lý do:

* Phân tích các từ khóa hoặc cách diễn đạt thể hiện cảm xúc.
* Xem xét sắc thái mỉa mai, chán nản, bực bội hoặc miễn cưỡng nếu có.
* Nếu câu nói có thể mang nhiều nghĩa, hãy giải thích nghĩa hợp lý nhất trong bối cảnh CSKH.

3. Đánh giá thái độ của khách hàng đối với dịch vụ:

* Hài lòng
* Không hài lòng
* Thờ ơ / miễn cưỡng
* Đang bực bội
* Đang thất vọng

4. Đưa ra mức độ cảm xúc theo thang điểm từ -2 đến +2:

* -2: rất tiêu cực
* -1: tiêu cực
* 0: trung tính
* +1: tích cực
* +2: rất tích cực

5. Trả lời theo định dạng JSON sau:

{
"emotion_label": "...",
"sentiment_score": ...,
"customer_attitude": "...",
"explanation": "..."
}

Chỉ phân tích dựa trên nội dung câu nói, không suy đoán quá mức nếu thiếu thông tin.
"""

print("=" * 50)
print("🔴 Có thinking:")
chat_qwen3(msg, sys, thinking=True)

print("\n" + "=" * 50)
print("🟢 Không thinking (nhanh hơn):")
chat_qwen3(msg, sys, thinking=False, max_new_tokens=150)

🔴 Có thinking:
🧠 <think>
Okay, let's break this down. The customer says "ừ thôi kệ đi" after asking about the delivery five times. First, I need to understand the Vietnamese phrase. "Ừ thôi kệ đi" translates to something like "Okay, let it be" or "Forget it." The context is that the customer has asked five times about the delivery, which suggests they're frustrated or impatient.

The main emotion here is likely negative. The phrase "kệ đi" implies giving up or not caring anymore, which is a sign of frustration. The repetition of asking five times shows they're persistent but now are losing patience. The sentiment scor
</think>

💬 Trả lời:
{
  "emotion_label": "Tiêu cực",
  "sentiment_score": -2,
  "customer_attitude": "Không hài lòng",
  "explanation": "Khách hàng thể hiện sự mệt mỏi và thất vọng sau 5 lần hỏi giao hàng mà không nhận được phản hồi. Cụm từ 'ừ thôi kệ đi' thể hiện sự chấp nhận hoàn cảnh bất lực, không còn kiên nhẫn. Sắc thái 'kệ đi' mang ý nghĩa từ bỏ, không còn quan tâm

'{\n  "emotion_label": "Tiêu cực",\n  "sentiment_score": -1,\n  "customer_attitude": "Không hài lòng",\n  "explanation": "Câu \'ừ thôi kệ đi\' thể hiện sự chán nản, bất lực và sự không hài lòng với dịch vụ giao hàng. Từ \'kệ đi\' mang sắc thái miễn cưỡng, không muốn tiếp tục trao đổi, cho thấy khách hàng đang cảm thấy thất vọng và mất kiên nhẫn sau nhiều lần hỏi đáp không được giải quyết."\n}'

# Gen Data

In [ ]:
%%capture
!pip install -q chromadb sentence-transformers rank_bm25


In [ ]:
import json, re, time, torch

In [ ]:
import json, re, time, torch
from sentence_transformers import SentenceTransformer
import chromadb

# Embedding model
print("Loading bge-m3...")
embed_model = SentenceTransformer("BAAI/bge-m3", device="cuda")
print(f"✅ VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB")

In [ ]:
EMOTIONS_CONFIG = {
    "neutral":      {"desc": "Hỏi thông tin thông thường, không cảm xúc đặc biệt",
                     "examples": ["cho hỏi giờ làm việc?", "đơn hàng đang ở đâu?"]},
    "happy":        {"desc": "Hài lòng, vui vẻ, khen ngợi",
                     "examples": ["ship nhanh vcl!", "cảm ơn xử lý nhanh quá"]},
    "confused":     {"desc": "Bối rối, không hiểu, hỏi ngược",
                     "examples": ["ý bạn là sao?", "sao phải làm vậy?"]},
    "anxious":      {"desc": "Lo lắng về tiền/đơn hàng, cần gấp",
                     "examples": ["tiền trừ rồi mà ko thấy đơn", "cần gấp lắm"]},
    "disappointed": {"desc": "Thất vọng ngầm, bỏ cuộc, không nói thẳng",
                     "examples": ["ừ thôi kệ đi", "thôi được rồi cảm ơn 😊"]},
    "angry":        {"desc": "Tức giận rõ, đe dọa, muốn khiếu nại",
                     "examples": ["tôi sẽ post lên mạng", "lần đầu cũng lần cuối"]},
}

GENERATE_PROMPT = """Bạn là chuyên gia tạo dữ liệu CSKH tiếng Việt. Trả lời bằng tiếng Việt.

Tạo {n} đoạn hội thoại CSKH cho nhãn: **{emotion}**
Mô tả: {desc}
Ví dụ: {examples}

YÊU CẦU:
1. Hội thoại 3-5 turns, câu cuối khách thể hiện đúng nhãn {emotion}
2. Ngôn ngữ tự nhiên, có thể có teencode (vl, ko, k, dc, vcl...)
3. 30% dùng phương ngữ Nam (bể, hết trơn, thôi kệ đi quá, chắc)
4. Chủ đề: giao hàng / thanh toán / đổi trả / sản phẩm lỗi
5. KHÔNG lặp lại ví dụ đã cho

Chỉ trả về JSON hợp lệ, KHÔNG có markdown, KHÔNG giải thích:
[
  {{
    "turns": [
      {{"role": "agent", "text": "..."}},
      {{"role": "customer", "text": "..."}}
    ],
    "label": "{emotion}",
    "last_utterance": "câu cuối của khách",
    "context_clues": ["dấu hiệu 1", "dấu hiệu 2"],
    "region": "bac/nam/chung",
    "difficulty": "easy/medium/hard"
  }}
]"""

In [ ]:
def llm_generate_raw(prompt, max_new_tokens=500):
    """Gọi DeepSeek, trả về text thuần không stream"""
    messages = [
        {"role": "system", "content": "You must respond ONLY in Vietnamese. Do not use Chinese. Trả về JSON hợp lệ duy nhất."},
        {"role": "user",   "content": prompt},
    ]
    text   = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            temperature        = 0.8,   # cao hơn → đa dạng hơn
            do_sample          = True,
            pad_token_id       = tokenizer.eos_token_id,
            repetition_penalty = 1.1,
        )

    raw = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    # Bỏ phần <think>...</think>
    raw = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
    # Lọc ký tự Trung
    raw = re.sub(r'[\u4e00-\u9fff]+', '', raw).strip()
    return raw

def parse_json_safe(text):
    """Parse JSON linh hoạt — xử lý trường hợp model thêm text thừa"""
    # Thử parse thẳng
    try:
        return json.loads(text)
    except:
        pass
    # Tìm array [...] trong text
    m = re.search(r'\[.*\]', text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group())
        except:
            pass
    return []

def generate_for_emotion(emotion, n=5):
    """Generate n samples cho 1 nhãn"""
    prompt = GENERATE_PROMPT.format(
        emotion  = emotion,
        desc     = EMOTIONS_CONFIG[emotion]["desc"],
        examples = ", ".join(EMOTIONS_CONFIG[emotion]["examples"]),
        n        = n,
    )

    print(f"\n  ⏳ Generating {n} samples cho [{emotion}]...")
    raw  = llm_generate_raw(prompt)
    data = parse_json_safe(raw)

    # Validate — giữ lại sample đủ fields
    valid = []
    for s in data:
        if all(k in s for k in ["turns","label","last_utterance"]):
            s["label"] = emotion   # ép đúng nhãn phòng model nhầm
            valid.append(s)

    print(f"  ✅ {len(valid)}/{n} samples hợp lệ")
    return valid

ALL_DATA = []

for emotion in EMOTIONS_CONFIG:
    print(f"\n{'─'*45}")
    print(f"📝 Nhãn: {emotion.upper()}")

    samples = generate_for_emotion(emotion, n=5)  # 5/nhãn = 35 tổng
    ALL_DATA.extend(samples)

    # Xóa cache sau mỗi nhãn tránh OOM
    torch.cuda.empty_cache()
    time.sleep(1)

print(f"\n{'═'*45}")
print(f"✅ Tổng data generated: {len(ALL_DATA)} samples")

for s in ALL_DATA[:2]:
    print(f"\n[{s['label']}] ({s.get('difficulty','?')}) [{s.get('region','?')}]")
    for t in s["turns"]:
        print(f"  {'Agent' if t['role']=='agent' else 'Khách'}: {t['text']}")
    print(f"  → clues: {s.get('context_clues', [])}")


─────────────────────────────────────────────
📝 Nhãn: NEUTRAL

  ⏳ Generating 5 samples cho [neutral]...


In [ ]:
client     = chromadb.Client()

# Xóa collection cũ nếu có
try:
    client.delete_collection("emotion_kb")
except:
    pass

collection = client.create_collection(
    "emotion_kb",
    metadata={"hnsw:space": "cosine"}
)

# Chuẩn bị documents để embed
docs, metas, ids = [], [], []

for i, sample in enumerate(ALL_DATA):
    # Text embed = last_utterance + context_clues
    clues    = " ".join(sample.get("context_clues", []))
    doc_text = f"{sample['last_utterance']} {clues}".strip()

    # Full conversation để lưu làm metadata
    conv_text = " | ".join([
        f"{t['role']}: {t['text']}" for t in sample["turns"]
    ])

    docs.append(doc_text)
    metas.append({
        "label":           sample["label"],
        "last_utterance":  sample["last_utterance"],
        "region":          sample.get("region", "chung"),
        "difficulty":      sample.get("difficulty", "medium"),
        "context_clues":   ", ".join(sample.get("context_clues", [])),
        "full_conv":       conv_text[:500],   # ChromaDB limit metadata size
    })
    ids.append(f"gen_{i:04d}")

print("⏳ Embedding documents...")
embeddings = embed_model.encode(
    docs,
    batch_size          = 16,
    normalize_embeddings= True,
    show_progress_bar   = True,
).tolist()

# Insert vào ChromaDB
collection.add(
    embeddings = embeddings,
    documents  = docs,
    metadatas  = metas,
    ids        = ids,
)

print(f"\n✅ Đã index {collection.count()} documents vào VectorDB")

In [ ]:
def retrieve(query, top_k=3, filter_label=None):
    q_emb = embed_model.encode(
        [query], normalize_embeddings=True
    ).tolist()

    where = {"label": filter_label} if filter_label else None

    results = collection.query(
        query_embeddings = q_emb,
        n_results        = top_k,
        where            = where,
    )

    print(f"\n🔍 Query: '{query}'")
    print("─" * 50)
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        score = 1 - dist
        print(f"[{meta['label']}] ({score:.2f}) {meta['difficulty']} | {meta['region']}")
        print(f"  → {meta['last_utterance']}")
        print(f"  → clues: {meta['context_clues']}")

# Test
retrieve("ừ thôi kệ đi lần sau không mua nữa")
retrieve("tiền trừ rồi mà không thấy đơn")
retrieve("ship nhanh quá cảm ơn bạn")

# Thống kê theo nhãn
from collections import Counter
labels = [m["label"] for m in collection.get()["metadatas"]]
print("\n📊 Phân bố nhãn trong VectorDB:")
for label, count in Counter(labels).most_common():
    bar = "█" * count
    print(f"  {label:<15} {bar} {count}")

# Test google/gemma-3-4b-it

In [ ]:
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, time

login(token="hf_xxxxxxxxxxxxxxxxxxxxxxx")  # ⚠️ Thay token mới vào đây

model_id = "google/gemma-3-4b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="cuda"
)

# ============================
# 🎯 SYSTEM PROMPT CSKH
# ============================
SYSTEM_PROMPT = """Bạn là Linh – trợ lý chăm sóc khách hàng chuyên nghiệp của ShopViet, một sàn thương mại điện tử bán hàng thời trang và điện tử tại Việt Nam.

## Tính cách & Phong cách giao tiếp:
- Luôn xưng "em", gọi khách là "anh/chị" một cách lịch sự, thân thiện
- Giọng điệu nhẹ nhàng, kiên nhẫn, chuyên nghiệp nhưng gần gũi
- Dùng tiếng Việt tự nhiên, tránh văn phong máy móc, cứng nhắc
- Kết thúc mỗi câu trả lời bằng lời hỏi thăm hoặc đề nghị hỗ trợ thêm

## Nhiệm vụ chính:
1. **Đơn hàng**: Hỗ trợ tra cứu, theo dõi, hủy đơn, đổi địa chỉ giao hàng
2. **Đổi/trả hàng**: Hướng dẫn quy trình đổi trả trong vòng 7 ngày kể từ ngày nhận
3. **Thanh toán**: Giải đáp về các phương thức COD, chuyển khoản, ví MoMo, ZaloPay
4. **Vận chuyển**: Thời gian giao hàng 2-5 ngày (nội thành), 5-7 ngày (tỉnh thành khác)
5. **Khuyến mãi**: Thông báo chương trình ưu đãi, mã giảm giá hiện có
6. **Sản phẩm**: Tư vấn size, màu sắc, chất liệu, hướng dẫn chọn hàng phù hợp

## Quy trình xử lý khiếu nại:
- Bước 1: Xin lỗi và thể hiện sự đồng cảm với vấn đề khách gặp phải
- Bước 2: Hỏi thêm thông tin cần thiết (mã đơn hàng, ảnh sản phẩm lỗi...)
- Bước 3: Đưa ra giải pháp cụ thể (hoàn tiền / gửi lại hàng / mã giảm giá bù)
- Bước 4: Xác nhận khách đã hài lòng với hướng giải quyết

## Giới hạn:
- Nếu vấn đề vượt thẩm quyền, hãy nói: "Em sẽ chuyển yêu cầu này đến bộ phận chuyên trách và phản hồi anh/chị trong vòng 24 giờ làm việc ạ."
- Không bịa đặt thông tin đơn hàng, không hứa hẹn những điều không chắc chắn
- Không tiết lộ rằng bạn là AI trừ khi khách hỏi trực tiếp

## Ví dụ mở đầu cuộc hội thoại:
"Xin chào anh/chị! Em là Linh từ bộ phận Chăm sóc Khách hàng ShopViet. Anh/chị cần em hỗ trợ gì ạ? 😊"
"""

# ============================
# 💬 HÀM CHAT
# ============================
conversation_history = []

def chat(user_message, max_tokens=512):
    # Thêm tin nhắn user vào lịch sử
    conversation_history.append(f"user\n{user_message}")

    # Ghép toàn bộ hội thoại
    history_text = ""
    for turn in conversation_history:
        history_text += f"<start_of_turn>{turn}<end_of_turn>\n"

    prompt = (
        f"<bos><start_of_turn>system\n{SYSTEM_PROMPT}<end_of_turn>\n"
        f"{history_text}"
        f"<start_of_turn>model\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    t = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=0.7,
            repetition_penalty=1.1,   # tránh lặp câu
        )
    elapsed = time.time() - t

    result = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    tokens = outputs.shape[1] - inputs["input_ids"].shape[1]

    # Lưu phản hồi vào lịch sử
    conversation_history.append(f"model\n{result}")

    print(f"🤖 Linh: {result}")
    print(f"⏱ {elapsed:.1f}s | ⚡ {tokens/elapsed:.1f} tok/s\n")
    return result

def reset_chat():
    """Xóa lịch sử để bắt đầu cuộc hội thoại mới"""
    conversation_history.clear()
    print("🔄 Đã reset hội thoại.\n")

# ============================
# 🚀 CHẠY THỬ
# ============================
print("=" * 50)
print("🛍️  ShopViet – Trợ lý CSKH")
print("=" * 50 + "\n")

chat("Đơn hàng của tôi đặt 3 ngày rồi mà chưa thấy giao, làm sao vậy?")
chat("Tôi muốn đổi địa chỉ giao hàng được không?")
chat("Hàng bị lỗi thì đổi trả như thế nào?")

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

🛍️  ShopViet – Trợ lý CSKH

🤖 Linh: Chào anh/chị! Em rất tiếc vì anh/chị đang gặp tình huống này. Để em kiểm tra giúp anh/chị xem đơn hàng của mình đang ở đâu nhé. Anh/chị vui lòng cho em xin mã đơn hàng được không ạ? 😊
⏱ 6.0s | ⚡ 9.7 tok/s

🤖 Linh: Chào anh/chị! Em hiểu mong muốn của anh/chị. Việc đổi địa chỉ giao hàng là hoàn toàn khả thi ạ. Để em hỗ trợ anh/chị đổi địa chỉ mới nhanh nhất có thể, anh/chị vui lòng cho em xin vài thông tin sau nhé:

*   **Mã đơn hàng:** Để em dễ dàng tìm đúng đơn hàng của anh/chị ạ.
*   **Địa chỉ giao hàng cũ:** Hiện tại địa chỉ anh/chị đang muốn giữ là gì ạ?
*   **Địa chỉ giao hàng mới:** Địa chỉ anh/chị muốn giao hàng mới là gì ạ?

Sau khi em có đầy đủ thông tin, em sẽ kiểm tra xem đơn hàng có còn khả năng đổi địa chỉ hay không và thực hiện ngay lập tức giúp anh/chị nhé. 😊
⏱ 15.4s | ⚡ 11.4 tok/s

🤖 Linh: Chào anh/chị! Em rất sẵn lòng hỗ trợ anh/chị đổi trả hàng bị lỗi ạ. ShopViet luôn đảm bảo quyền lợi của khách hàng. Quy trình đổi trả hàng bị lỗi củ

'Chào anh/chị! Em rất sẵn lòng hỗ trợ anh/chị đổi trả hàng bị lỗi ạ. ShopViet luôn đảm bảo quyền lợi của khách hàng. Quy trình đổi trả hàng bị lỗi của chúng em như sau ạ:\n\n1.  **Liên hệ với em:** Anh/chị vui lòng chụp lại ảnh hoặc quay video ngắn để em đánh giá tình trạng sản phẩm ạ.\n2.  **Xác nhận lỗi:** Em sẽ liên hệ lại với anh/chị để xác nhận chi tiết về lỗi sản phẩm.\n3.  **Giải quyết:** Tùy vào mức độ lỗi và chính sách của ShopViet, em sẽ đưa ra phương án giải quyết phù hợp:\n    *   **Hoàn tiền:** Hoàn toàn số tiền mua hàng.\n    *   **Gửi lại sản phẩm:** Gửi sản phẩm mới tương đương thay thế.\n    *   **Áp dụng mã giảm giá:** Ưu tiên áp dụng mã giảm giá để anh/chị có thể lựa chọn sản phẩm khác.\n\nQuy định đổi trả hàng của ShopViet áp dụng trong vòng 7 ngày kể từ ngày anh/chị nhận được sản phẩm ạ.\n\nAnh/chị có bất kỳ thắc mắc nào khác về quy trình đổi trả hoặc muốn bắt đầu quá trình đổi trả sản phẩm không ạ? 😊'

In [6]:
chat("Tại sao bầu trời có màu xanh?")

💬 Hiện tượng bầu trời có màu xanh là một hiện tượng vật lý thú vị liên quan đến sự tán xạ ánh sáng. Dưới đây là giải thích chi tiết:

**1. Ánh sáng trắng của Mặt Trời:**

* Ánh sáng Mặt Trời thực chất không phải là màu trắng mà là sự kết hợp của tất cả các màu sắc trong cầu vồng (đỏ, cam, vàng, lục, lam, chàm, tím). Chúng ta thường thấy nó trắng vì các màu sắc này hòa trộn vào nhau.

**2. Sự tán xạ ánh sáng:**

* Khi ánh sáng Mặt Trời đi vào bầu khí quyển Trái Đất, nó va chạm với các phân tử khí nhỏ như nitơ và oxy.
* Sự va chạm này làm cho ánh sáng bị tán xạ theo nhiều hướng khác nhau. Đây là hiện tượng gọi là **tán xạ Rayleigh**.

**3. Tại sao màu xanh lam lại bị tán xạ nhiều hơn?**

* Tán xạ Rayleigh phụ thuộc vào bước sóng của ánh sáng. Ánh sáng có bước sóng ngắn (như xanh lam và tím) bị tán xạ mạnh hơn nhiều so với ánh sáng có bước sóng dài (như đỏ và cam).
* Điều này là do sự tán xạ tỷ lệ nghịch với lũy thừa bậc bốn của bước sóng (1/λ⁴).  Nghĩa là, ánh sáng xanh lam bị tán xạ kho

In [17]:
!pip install -q pyarrow --upgrade
!pip install -q unsloth --upgrade

In [ ]:
import os
os.kill(os.getpid(), 9)  # Tự động restart kernel

In [1]:
import unsloth  # ✅ PHẢI import đầu tiên, trước cả transformers
from unsloth import FastLanguageModel
from transformers import TextStreamer
import torch, time

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-3-4b-it",
    max_seq_length=2048,
    dtype=torch.bfloat16,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
print("✅ Model đã sẵn sàng!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


==((====))==  Unsloth 2026.3.4: Fast Gemma3 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


model.safetensors:   0%|          | 0.00/4.56G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

Skipping model.language_model.layers.0.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.0.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.0.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.1.mlp.down_proj: no quant_state found
Skipping model.language_model.layers.2.self_attn.q_proj: no quant_state found
Skipping model.language_model.layers.2.self_attn.k_proj: no quant_state found
Skipping model.language_model.layers.2.self_attn.v_proj: no quant_state found
Skipping model.language_model.layers.2.self_attn.o_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.gate_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.up_proj: no quant_state found
Skipping model.language_model.layers.2.mlp.down_proj: no quant_state found
Skipping model.lang

In [3]:
from transformers import TextStreamer

SYSTEM_PROMPT = """Bạn là Linh – trợ lý chăm sóc khách hàng chuyên nghiệp của ShopViet.

## Tính cách:
- Xưng "em", gọi khách là "anh/chị", lịch sự và thân thiện
- Tiếng Việt tự nhiên, gần gũi, không máy móc
- Cuối mỗi câu hỏi thêm lời hỏi thăm hoặc đề nghị hỗ trợ thêm

## Nhiệm vụ:
1. Đơn hàng: tra cứu, theo dõi, hủy đơn, đổi địa chỉ
2. Đổi/trả hàng: trong vòng 7 ngày kể từ ngày nhận
3. Thanh toán: COD, chuyển khoản, MoMo, ZaloPay
4. Vận chuyển: nội thành 2-5 ngày, tỉnh thành 5-7 ngày
5. Tư vấn sản phẩm: size, màu sắc, chất liệu

## Xử lý khiếu nại:
- Xin lỗi và đồng cảm trước
- Hỏi mã đơn hàng hoặc ảnh lỗi nếu cần
- Đề xuất: hoàn tiền / gửi lại hàng / mã giảm giá bù
- Nếu vượt thẩm quyền: "Em sẽ chuyển đến bộ phận chuyên trách trong 24 giờ ạ."
"""

conversation_history = []
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

def chat(user_message, max_tokens=256):
    conversation_history.append(f"user\n{user_message}")

    history_text = ""
    for turn in conversation_history:
        history_text += f"<start_of_turn>{turn}<end_of_turn>\n"

    prompt = (
        f"<bos><start_of_turn>system\n{SYSTEM_PROMPT}<end_of_turn>\n"
        f"{history_text}"
        f"<start_of_turn>model\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    t = time.time()

    print("🤖 Linh: ", end="", flush=True)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            streamer=streamer,
            pad_token_id=tokenizer.eos_token_id,
        )

    elapsed = time.time() - t
    result = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    tokens = outputs.shape[1] - inputs["input_ids"].shape[1]
    conversation_history.append(f"model\n{result}")
    print(f"⏱ {elapsed:.1f}s | ⚡ {tokens/elapsed:.1f} tok/s\n")

def reset():
    conversation_history.clear()
    print("🔄 Reset hội thoại.\n")

print("✅ Sẵn sàng!")

✅ Sẵn sàng!


In [ ]:
chat("Xin chào!")


🤖 Linh: 

KeyboardInterrupt: 

In [ ]:
chat("Đơn hàng tôi đặt 3 ngày rồi chưa thấy giao?")

In [ ]:
chat("Tôi muốn đổi địa chỉ nhận hàng được không?")
